# Correlación entre variables

Este notebook explora los conceptos de correlación lineal, no lineal y la diferencia entre independencia y correlación cero.  
Se comparan los coeficientes de **Pearson** y **Spearman** en cada escenario, y se incluye una celda interactiva para experimentar con outliers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

np.random.seed(42)

# Estilo general
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("Librerías cargadas correctamente.")

---
## Función auxiliar

Calculamos Pearson y Spearman y los mostramos en el título de cada gráfico.

In [ ]:
def pearson_spearman(x, y):
    """Devuelve (r_pearson, p_pearson, rho_spearman, p_spearman)."""
    r, p_r   = stats.pearsonr(x, y)
    rho, p_s = stats.spearmanr(x, y)
    return r, p_r, rho, p_s

def scatter_con_corr(ax, x, y, titulo, color='steelblue', alpha=0.6):
    """Scatter plot con Pearson y Spearman en el título."""
    r, _, rho, _ = pearson_spearman(x, y)
    ax.scatter(x, y, color=color, alpha=alpha, edgecolors='white', linewidths=0.4, s=50)
    ax.set_title(f"{titulo}\nPearson r = {r:.3f}   |   Spearman ρ = {rho:.3f}", fontsize=10, pad=8)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')

---
## 1. Correlación lineal

Generamos tres casos: correlación positiva fuerte, negativa fuerte y débil/nula lineal.

In [ ]:
n = 200
x = np.random.uniform(-3, 3, n)

# Positiva fuerte
y_pos = 2 * x + np.random.normal(0, 0.5, n)

# Negativa fuerte
y_neg = -1.5 * x + np.random.normal(0, 0.6, n)

# Débil
y_deb = 0.3 * x + np.random.normal(0, 2.5, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
scatter_con_corr(axes[0], x, y_pos, 'Correlación lineal positiva fuerte', color='#2196F3')
scatter_con_corr(axes[1], x, y_neg, 'Correlación lineal negativa fuerte', color='#E91E63')
scatter_con_corr(axes[2], x, y_deb, 'Correlación lineal débil',           color='#FF9800')

fig.suptitle('Correlación lineal — Pearson y Spearman coinciden', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observación:** Cuando la relación es lineal, Pearson y Spearman dan valores muy similares.  
Pearson mide correlación lineal directamente; Spearman lo hace sobre los rangos, lo que en datos sin outliers produce resultados análogos.

---
## 2. Correlación no lineal

Cuando la relación es monótona pero no lineal, Spearman la captura mejor que Pearson.

In [ ]:
x_pos = np.random.uniform(0.1, 5, n)

# Monotona: log
y_log = np.log(x_pos) + np.random.normal(0, 0.3, n)

# Monotona: exponencial
y_exp = np.exp(0.5 * x_pos) + np.random.normal(0, 1.5, n)

# No monótona: cuadrática (U invertida)
x_cuad = np.random.uniform(-3, 3, n)
y_cuad = -x_cuad**2 + np.random.normal(0, 0.8, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
scatter_con_corr(axes[0], x_pos,  y_log,  'Relación logarítmica (monótona)',    color='#4CAF50')
scatter_con_corr(axes[1], x_pos,  y_exp,  'Relación exponencial (monótona)',    color='#9C27B0')
scatter_con_corr(axes[2], x_cuad, y_cuad, 'Relación cuadrática (no monótona)',  color='#F44336')

fig.suptitle('Correlación no lineal — Spearman vs Pearson', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observación:**  
- En relaciones **monótonas** (log, exp), Spearman detecta la dependencia mejor que Pearson.  
- En la relación **cuadrática simétrica** (U invertida), ambos coeficientes son cercanos a cero: la relación es fuerte pero ni Pearson ni Spearman la capturan, porque ninguno mide dependencia en forma de U.  
  → Para eso necesitaríamos otras herramientas (información mutua, distancia de correlación, etc.).

---
## 3. Independencia implica correlación 0, pero no al revés

Mostramos tres situaciones con correlación ≈ 0:  
1. Variables **genuinamente independientes**  
2. Variables con **dependencia no lineal simétrica** (correlación 0 pero no independientes)  
3. Variables con **patrón circular** (dependencia evidente, correlación 0)

In [ ]:
# 1. Independientes
x_ind = np.random.normal(0, 1, n)
y_ind = np.random.normal(0, 1, n)

# 2. Dependencia no lineal simétrica: Y = X^2 + ruido
x_nl = np.random.uniform(-3, 3, n)
y_nl = x_nl**2 + np.random.normal(0, 0.5, n)

# 3. Patrón circular
theta = np.random.uniform(0, 2 * np.pi, n)
x_circ = np.cos(theta) + np.random.normal(0, 0.1, n)
y_circ = np.sin(theta) + np.random.normal(0, 0.1, n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
scatter_con_corr(axes[0], x_ind,  y_ind,  'Variables independientes',           color='#607D8B')
scatter_con_corr(axes[1], x_nl,   y_nl,   'Y = X² + ruido  (dep. no lineal)',   color='#FF5722')
scatter_con_corr(axes[2], x_circ, y_circ, 'Patrón circular',                    color='#00BCD4')

fig.suptitle('Correlación ≈ 0 NO implica independencia', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observación:**  
Los tres gráficos muestran correlación ≈ 0, pero solo el primero corresponde a variables **independientes**.  
En los otros dos hay una relación clara visible en la nube de puntos.  

> **Regla de oro:** independencia ⟹ correlación 0, pero correlación 0 ⟹̸ independencia.

---
## 4. Efecto de outliers sobre Pearson y Spearman

Pearson es muy sensible a outliers porque trabaja con los valores originales.  
Spearman es robusto porque trabaja con rangos.

**Instrucciones para jugar:**  
Modificá las variables `n_outliers`, `outlier_x` y `outlier_y` para ver cómo cambian ambos coeficientes.

In [ ]:
# ============================================================
#  PARÁMETROS — MODIFICAR PARA EXPLORAR
# ============================================================
n_base      = 100     # Tamaño de la muestra base
n_outliers  = 3       # Cantidad de outliers a agregar
outlier_x   = 10.0    # Posición X de los outliers
outlier_y   = 10.0    # Posición Y de los outliers
#  Probá también: outlier_x=10, outlier_y=-10 (outlier en dirección opuesta)
#                 n_outliers=1, outlier_x=5, outlier_y=5
# ============================================================

# Datos base: correlación baja / nula
x_base = np.random.normal(0, 1, n_base)
y_base = np.random.normal(0, 1, n_base)

# Agregamos outliers
x_out = np.concatenate([x_base, np.full(n_outliers, outlier_x)])
y_out = np.concatenate([y_base, np.full(n_outliers, outlier_y)])

# Cálculo de correlaciones
r_base,   _, rho_base,   _ = pearson_spearman(x_base, y_base)
r_out,    _, rho_out,    _ = pearson_spearman(x_out,  y_out)

# Gráficos
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sin outliers
axes[0].scatter(x_base, y_base, color='steelblue', alpha=0.6, edgecolors='white', linewidths=0.4, s=50)
axes[0].set_title(f"Sin outliers\nPearson r = {r_base:.3f}   |   Spearman ρ = {rho_base:.3f}", fontsize=10)
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')

# Con outliers
axes[1].scatter(x_base, y_base, color='steelblue', alpha=0.5, edgecolors='white', linewidths=0.4, s=50, label='Datos base')
axes[1].scatter(np.full(n_outliers, outlier_x), np.full(n_outliers, outlier_y),
                color='red', s=120, zorder=5, edgecolors='black', linewidths=0.8, label='Outliers')
axes[1].set_title(f"Con {n_outliers} outlier(s) en ({outlier_x}, {outlier_y})\nPearson r = {r_out:.3f}   |   Spearman ρ = {rho_out:.3f}", fontsize=10)
axes[1].set_xlabel('X'); axes[1].set_ylabel('Y')
axes[1].legend()

fig.suptitle('Efecto de outliers: Pearson vs Spearman', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nCambio en Pearson:  {r_base:.3f}  →  {r_out:.3f}   (Δ = {r_out - r_base:+.3f})")
print(f"Cambio en Spearman: {rho_base:.3f}  →  {rho_out:.3f}   (Δ = {rho_out - rho_base:+.3f})")

**Para experimentar — sugerencias:**

| Configuración | Qué observar |
|---|---|
| `outlier_x=10, outlier_y=10, n_outliers=1` | Un solo outlier puede inflar Pearson notablemente |
| `outlier_x=10, outlier_y=-10` | Outlier en dirección opuesta: Pearson se vuelve negativo |
| `n_outliers=5, outlier_x=3, outlier_y=3` | Outliers moderados: Spearman apenas se mueve |
| `n_outliers=1, outlier_x=0, outlier_y=8` | Outlier vertical: mínimo efecto en Pearson |

> **Conclusión:** Pearson es sensible a valores extremos. Spearman, al basarse en rangos, los absorbe sin distorsionarse significativamente. Cuando hay sospechas de outliers o distribuciones de cola pesada, Spearman es la opción más robusta.

---
## 5. Resumen comparativo

Panel con los escenarios principales para tener todo en un solo vistazo.

In [ ]:
escenarios = [
    ('Lineal positiva',         x,      y_pos,  '#2196F3'),
    ('Lineal negativa',         x,      y_neg,  '#E91E63'),
    ('Logarítmica',             x_pos,  y_log,  '#4CAF50'),
    ('Cuadrática (U inv.)',     x_cuad, y_cuad, '#F44336'),
    ('Independientes',          x_ind,  y_ind,  '#607D8B'),
    ('Y = X² (corr 0, dep.)',  x_nl,   y_nl,   '#FF5722'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (titulo, xi, yi, color) in enumerate(escenarios):
    scatter_con_corr(axes[i], xi, yi, titulo, color=color)

fig.suptitle('Panel resumen — Pearson r y Spearman ρ en distintos escenarios',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()